# **Coffee Bean Dataset: Pre-Process**


1. Load dataset (reuse cache)

In [101]:
import os
import kagglehub

# Get dataset path (uses cache)
path = kagglehub.dataset_download("gpiosenka/coffee-bean-dataset-resized-224-x-224")

data_dir = os.path.join(path, "train")
classes = sorted(os.listdir(data_dir))

print(classes)

['Dark', 'Green', 'Light', 'Medium']


2. Collect image paths + labels

In [102]:
image_paths = []
labels = []

for cls in classes:
    cls_path = os.path.join(data_dir, cls)
    for img_name in os.listdir(cls_path):
        image_paths.append(os.path.join(cls_path, img_name))
        labels.append(cls)

3. Remove duplicates

In [103]:
import hashlib

def file_hash(path):
    with open(path, 'rb') as f:
        return hashlib.md5(f.read()).hexdigest()

unique = {}
clean_paths = []
clean_labels = []

for path, label in zip(image_paths, labels):
    h = file_hash(path)
    if h not in unique:
        unique[h] = True
        clean_paths.append(path)
        clean_labels.append(label)

print("Total after dedup:", len(clean_paths))

Total after dedup: 1194


4. Train / Validation split

In [104]:
from sklearn.model_selection import train_test_split

X_train, X_val, y_train, y_val = train_test_split(
    clean_paths,
    clean_labels,
    test_size=0.2,
    stratify=clean_labels,
    random_state=42
)

5. Transformations
Based on EDA:

- keep color signal
- small augmentation
- avoid distortion

In [105]:
from torchvision import transforms

train_transform = transforms.Compose([
    transforms.Resize((224, 224)),

    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(10),

    transforms.ColorJitter(
        brightness=0.2,
        contrast=0.2,
        saturation=0.2
    ),

    transforms.ToTensor(),

    transforms.Normalize(
        mean=[0.5, 0.5, 0.5],
        std=[0.5, 0.5, 0.5]
    )
])

In [106]:
#Validation transforms
val_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize([0.5]*3, [0.5]*3)
])

6. Label encoding

In [107]:
from sklearn.preprocessing import LabelEncoder

le = LabelEncoder()
le.fit(y_train)

LabelEncoder()

7. Custom Dataset

In [108]:
from torch.utils.data import Dataset
from PIL import Image

class CoffeeDataset(Dataset):
    def __init__(self, paths, labels, transform, label_encoder):
        self.paths = paths
        self.labels = labels
        self.transform = transform
        self.le = label_encoder

    def __len__(self):
        return len(self.paths)

    def __getitem__(self, idx):
        img = Image.open(self.paths[idx]).convert("RGB")
        label = self.le.transform([self.labels[idx]])[0]

        img = self.transform(img)
        return img, label

8. DataLoaders

In [109]:
from torch.utils.data import DataLoader

train_dataset = CoffeeDataset(X_train, y_train, train_transform, le)
val_dataset = CoffeeDataset(X_val, y_val, val_transform, le)

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=32, shuffle=False)

In [110]:
# sanity check
images, labels = next(iter(train_loader))

print(images.shape)  # should be [32, 3, 224, 224]
print(labels[:5])

torch.Size([32, 3, 224, 224])
tensor([0, 2, 1, 3, 0])


### Preprocessing

The dataset was preprocessed based on insights from the EDA:

- Duplicate images were removed to prevent data leakage
- A stratified 80/20 train-validation split was applied
- Images were normalized while preserving brightness and color differences
- Moderate data augmentation (rotation, flipping, color jitter) was applied to improve generalization

These steps ensure that the model learns meaningful visual features without distorting key signals.

In [111]:
#SAVE SPLIT FOR TRAINING
import pickle
from sklearn.preprocessing import LabelEncoder

le = LabelEncoder()
le.fit(y_train)

# you already have this:
with open("coffee_splits.pkl", "wb") as f:
    pickle.dump((X_train, X_val, y_train, y_val), f)

# add this:
with open("label_encoder.pkl", "wb") as f:
    pickle.dump(le, f)